# Timonel SFT — Piloto Llama 3.1 8B con Unsloth + QLoRA**Objetivo:** entrenar Llama 3.1 8B para discriminar acciones de navegacion segun el estado del barco.**Preregistro:** EXPEDIENTE/82_PREREGISTRO_SFT.md**Dataset:** 240 train + 60 val desde banco CC002 desarrollo.**Criterio de exito:** accuracy en val > 25% (baseline zero-shot), uso de las 4 acciones.

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets

## 2. Cargar modelo base con QLoRA

In [ ]:
from unsloth import FastLanguageModelimport torchmax_seq_length = 2048dtype = Noneload_in_4bit = Truemodel, tokenizer = FastLanguageModel.from_pretrained(    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",    max_seq_length = max_seq_length,    dtype = dtype,    load_in_4bit = load_in_4bit,)

## 3. Configurar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r = 16,    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",                      "gate_proj", "up_proj", "down_proj"],    lora_alpha = 32,    lora_dropout = 0.05,    bias = "none",    use_gradient_checkpointing = "unsloth",    random_state = 3407,    use_rslora = False,    loftq_config = None,)

## 4. Cargar el dataset (subir train.jsonl y val.jsonl)

In [ ]:
from datasets import load_datasetimport json# El dataset debe estar subido a /kaggle/input/timonel-sft/train.jsonl# o subido como Dataset en Kaggle.# Ajustar el path segun como se suba.TRAIN_PATH = "/kaggle/input/timonel-sft/train.jsonl"VAL_PATH = "/kaggle/input/timonel-sft/val.jsonl"dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")dataset_val = load_dataset("json", data_files=VAL_PATH, split="train")print(f"Train: {len(dataset_train)} | Val: {len(dataset_val)}")print(dataset_train[0])

## 5. Aplicar chat template

In [ ]:
from unsloth import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")def formatear(ejemplo):    texto = tokenizer.apply_chat_template(        ejemplo["messages"],        tokenize=False,        add_generation_prompt=False,    )    return {"text": texto}dataset_train = dataset_train.map(formatear)dataset_val = dataset_val.map(formatear)print("=== Ejemplo formateado ===")print(dataset_train[0]["text"][:800])

## 6. Configurar y correr el entrenamiento

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedtrainer = SFTTrainer(    model = model,    tokenizer = tokenizer,    train_dataset = dataset_train,    eval_dataset = dataset_val,    dataset_text_field = "text",    max_seq_length = max_seq_length,    dataset_num_proc = 2,    packing = False,    args = TrainingArguments(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,        warmup_steps = 5,        max_steps = 180,  # 3 epochs x 60 steps aprox        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 10,        eval_strategy = "steps",        eval_steps = 60,        save_strategy = "steps",        save_steps = 60,        optim = "adamw_8bit",        weight_decay = 0.01,        lr_scheduler_type = "linear",        seed = 3407,        output_dir = "outputs",        report_to = "none",    ),)stats = trainer.train()print(stats)

## 7. Guardar el adaptador LoRA

In [ ]:
model.save_pretrained("timonel_lora")tokenizer.save_pretrained("timonel_lora")# Opcional: guardar como GGUF para Ollama# model.save_pretrained_gguf("timonel_gguf", tokenizer, quantization_method="q4_k_m")print("Adaptador guardado en timonel_lora/")!ls -la timonel_lora/

## 8. Evaluar en validacionMetrica: accuracy de accion, distribucion de uso.

In [ ]:
import reimport jsonfrom collections import CounterFastLanguageModel.for_inference(model)def extraer_json(texto):    i = texto.find("{")    j = texto.rfind("}")    if i == -1 or j == -1 or j <= i:        return None    try:        return json.loads(texto[i:j+1])    except Exception:        return Nonedef predecir(messages_user_system):    inputs = tokenizer.apply_chat_template(        messages_user_system,        tokenize=True,        add_generation_prompt=True,        return_tensors="pt",    ).to("cuda")    with torch.no_grad():        out = model.generate(            input_ids=inputs,            max_new_tokens=64,            temperature=0.0,            do_sample=False,            use_cache=True,        )    respuesta = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)    return respuestaaciertos = 0total = 0uso = Counter()por_accion_ok = Counter()por_accion_tot = Counter()for ejemplo in dataset_val:    msgs = ejemplo["messages"]    user_sys = [msgs[0], msgs[1]]    respuesta_real = json.loads(msgs[2]["content"])    crudo = predecir(user_sys)    parsed = extraer_json(crudo)    pred = parsed.get("accion") if parsed else "_error_"    true = respuesta_real["accion"]    uso[pred] += 1    por_accion_tot[true] += 1    if pred == true:        aciertos += 1        por_accion_ok[true] += 1    total += 1print(f"=== RESULTADO VALIDACION ===")print(f"Accuracy: {aciertos}/{total} = {100*aciertos/total:.1f}%")print(f"Uso de acciones: {dict(uso)}")print(f"Por accion: {dict(por_accion_ok)} / {dict(por_accion_tot)}")